In [4]:
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon

# === INPUT FILES ===
ede_boundary_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\Ede\Ede_shape.shp"             # Boundary shapefile
tick_bites_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_bites_ede.gpkg"          # Point layer with tick bites
output_grid_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\hex_grid_ede.gpkg"         # Output hex grid with tick densities

# === PARAMETERS ===
hex_size = 500  # meters (distance from center to side of hexagon)

# === LOAD LAYERS ===
ede_boundary = gpd.read_file(ede_boundary_path)
tick_bites = gpd.read_file(tick_bites_path)

# Reproject to projected CRS (for meter-based grid)
# Ensure CRS is set (assume RD New for Netherlands if missing)
if ede_boundary.crs is None:
    ede_boundary.set_crs(epsg=28992, inplace=True)

# Reproject to RD New
ede_boundary = ede_boundary.to_crs(epsg=28992)
tick_bites = tick_bites.to_crs(ede_boundary.crs)

# === COMPUTE BOUNDS ===
minx, miny, maxx, maxy = ede_boundary.total_bounds

# === HEX GRID DIMENSIONS ===
radius = hex_size  # distance from center to vertex
dx = 1.5 * radius  # horizontal spacing between hex centers
dy = np.sqrt(3) * radius  # vertical spacing between hex centers

# === GENERATE HEX GRID ===
polygons = []
x = minx
row = 0
while x < maxx + dx:
    y = miny
    while y < maxy + dy:
        # Offset every other row
        cx = x
        cy = y + (dy / 2 if row % 2 else 0)

        # Create hexagon around center (cx, cy)
        coords = [(cx + radius * np.cos(np.radians(angle)),
                   cy + radius * np.sin(np.radians(angle)))
                  for angle in range(0, 360, 60)]
        hex_poly = Polygon(coords)

        if hex_poly.intersects(ede_boundary.unary_union):
            polygons.append(hex_poly)
        y += dy
    x += dx
    row += 1

hex_grid = gpd.GeoDataFrame(geometry=polygons, crs=ede_boundary.crs)

# === CLIP TO EDE BOUNDARY ===
hex_grid = gpd.overlay(hex_grid, ede_boundary, how='intersection')

# === SPATIAL JOIN: COUNT TICK BITES PER HEX ===
joined = gpd.sjoin(tick_bites, hex_grid, how="left", predicate="within")
tick_counts = joined.groupby("index_right").size()

hex_grid["tick_count"] = hex_grid.index.map(tick_counts).fillna(0).astype(int)

# === NORMALIZE TO DENSITY 0.1 to 1.0 (OPTIONAL) ===
min_count = hex_grid["tick_count"].min()
max_count = hex_grid["tick_count"].max()
hex_grid["tick_density"] = hex_grid["tick_count"].apply(
    lambda x: 0.1 * (x - min_count) / (max_count - min_count) if max_count != min_count else 0.0
)

# === SAVE OUTPUT ===
hex_grid.to_file(output_grid_path)
print(f"✅ Hex grid saved to: {output_grid_path}")


C:\Users\arman\AppData\Local\Temp\ipykernel_22696\3503083148.py:51: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  if hex_poly.intersects(ede_boundary.unary_union):


✅ Hex grid saved to: C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\hex_grid_ede.gpkg


In [14]:
import geopandas as gpd

ede_boundary = gpd.read_file(r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\Ede\Ede_shape.shp").set_crs(epsg=28992)
minx, miny, maxx, maxy = ede_boundary.total_bounds

width = maxx - minx
height = maxy - miny

print(f"Ede width: {width:.1f} m, height: {height:.1f} m")

resolution_x = width / 33
print(resolution_x)
resolution_y = height / 33
print(resolution_y)

Ede width: 27733.6 m, height: 17975.1 m
840.4117265181117
544.6988173976089


In [12]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin

# === INPUT AND OUTPUT PATHS ===
hex_grid_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\hex_grid_ede.gpkg"
tif_output_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede.tif"
asc_output_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede.asc"

# === PARAMETERS ===
resolution_x = 840.4117  # meters per pixel in X (from NetLogo)
resolution_y = 544.6988  # meters per pixel in Y (from NetLogo)

# === LOAD HEX GRID ===
gdf = gpd.read_file(hex_grid_path)

# Ensure CRS is projected (e.g., EPSG:28992)
if gdf.crs is None:
    raise ValueError("Hex grid has no CRS defined.")

# Get bounds
minx, miny, maxx, maxy = gdf.total_bounds

# Calculate raster dimensions (height = number of rows, width = number of columns)
width = int((maxx - minx) / resolution_x) + 1  # add 1 to cover full extent
height = int((maxy - miny) / resolution_y) + 1

# Create transform (top-left corner, pixel size in x and y)
transform = from_origin(minx, maxy, resolution_x, resolution_y)

# Create shapes (geometry, value) pairs for rasterization
shapes = ((geom, value) for geom, value in zip(gdf.geometry, gdf["tick_density"]))

# Rasterize vector shapes into numpy array
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0.0,  # value for areas outside polygons
    dtype="float32"
)

# Save raster as GeoTIFF
with rasterio.open(
    tif_output_path,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype='float32',
    crs=gdf.crs,
    transform=transform
) as dst:
    dst.write(raster, 1)

print(f"✅ Raster saved to: {tif_output_path}")

# === EXPORT TO ASCII GRID (.asc) ===
with rasterio.open(tif_output_path) as src:
    data = src.read(1)
    profile = src.profile
    profile.update(driver='AAIGrid')
    with rasterio.open(asc_output_path, 'w', **profile) as asc_dst:
        asc_dst.write(data, 1)

print(f"✅ ASCII Grid saved to: {asc_output_path}")

✅ Raster saved to: C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede.tif
✅ ASCII Grid saved to: C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede.asc


In [19]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin

# === INPUT AND OUTPUT PATHS ===
hex_grid_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\hex_grid_ede.gpkg"
tif_output_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede.tif"
asc_output_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede.asc"

# === PARAMETERS ===
resolution_x = 840.4117  # meters per pixel in X (from NetLogo)
resolution_y = 544.6988  # meters per pixel in Y (from NetLogo)

# === LOAD HEX GRID ===
gdf = gpd.read_file(hex_grid_path)

# Ensure CRS is projected
if gdf.crs is None:
    raise ValueError("Hex grid has no CRS defined.")

# === REPROJECT TO NETLOGO-COMPATIBLE CRS (EPSG:28992 assumed — adjust if needed) ===
target_crs = "EPSG:28992"  # Change to 'EPSG:4326' if NetLogo uses lat/lon degrees
if gdf.crs.to_string() != target_crs:
    gdf = gdf.to_crs(target_crs)

# Get bounds
minx, miny, maxx, maxy = gdf.total_bounds

# Calculate raster dimensions
width = int((maxx - minx) / resolution_x) + 1
height = int((maxy - miny) / resolution_y) + 1

# Create transform
transform = from_origin(minx, maxy, resolution_x, resolution_y)

# Create shapes (geometry, value) pairs
shapes = ((geom, value) for geom, value in zip(gdf.geometry, gdf["tick_density"]))

# Rasterize
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0.0,
    dtype="float32"
)

# Save as GeoTIFF
with rasterio.open(
    tif_output_path,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype='float32',
    crs=target_crs,
    transform=transform
) as dst:
    dst.write(raster, 1)

print(f"✅ Raster saved to: {tif_output_path}")

# === EXPORT TO ASCII GRID (.asc) ===
with rasterio.open(tif_output_path) as src:
    data = src.read(1)
    profile = src.profile

    # Update for AAIGrid export: remove CRS since .asc doesn't store it
    profile.update({
        'driver': 'AAIGrid',
        'crs': None,
        'dtype': 'float32',
        'count': 1,
        'transform': transform,
        'nodata': 0.0
    })

    with rasterio.open(asc_output_path, 'w', **profile) as asc_dst:
        asc_dst.write(data, 1)

print(f"✅ ASCII Grid saved to: {asc_output_path}")

✅ Raster saved to: C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede.tif
✅ ASCII Grid saved to: C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede.asc


In [17]:
import geopandas as gpd
import pandas as pd

# Load Ede land use shapefile
ede_landuse = gpd.read_file(r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\Ede\Ede_lnduse.shp")

# If CRS is missing, assign it (assuming RD New)
if ede_landuse.crs is None:
    ede_landuse.set_crs("EPSG:28992", inplace=True)

# Filter only relevant BG2015 values
ede_landuse = ede_landuse[ede_landuse["BG2015"].isin([20, 60, 61, 62])]

# Load tick bites point shapefile
tick_bites = gpd.read_file(r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\NL_TickBites_Dec16_RD_New.shp")

# Ensure CRS matches
if ede_landuse.crs != tick_bites.crs:
    tick_bites = tick_bites.to_crs(ede_landuse.crs)

# Spatial join: assign each tick bite to its land use polygon in Ede
joined = gpd.sjoin(tick_bites, ede_landuse, how="inner", predicate="within")

# Calculate area in square kilometers for each land use category
ede_landuse["area_km2"] = ede_landuse.geometry.area / 1e6

# Count tick bites per BG2015 category
tick_counts = joined.groupby("BG2015").size().reset_index(name="tick_count")

# Sum area per BG2015 category
area_sums = ede_landuse.groupby("BG2015")["area_km2"].sum().reset_index()

# Merge tick counts and area sums
summary = pd.merge(tick_counts, area_sums, on="BG2015", how="left")

# Compute raw tick density (tick bites per km²)
summary["tick_density_per_km2"] = summary["tick_count"] / summary["area_km2"]

# Min-max normalize tick density to 0–1
min_density = summary["tick_density_per_km2"].min()
max_density = summary["tick_density_per_km2"].max()
summary["normalized_density"] = (summary["tick_density_per_km2"] - min_density) / (max_density - min_density)

# Print final result
print(summary)

   BG2015  tick_count    area_km2  tick_density_per_km2  normalized_density
0      20         238   14.122593             16.852430            1.000000
1      60         429  110.255249              3.890971            0.121986
2      61         150   56.934330              2.634614            0.036880
3      62         287  137.308252              2.090188            0.000000


In [24]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin

# === INPUT AND OUTPUT PATHS ===
hex_grid_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\hex_grid_ede.gpkg"
tif_output_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede_50m.tif"
asc_output_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede_50m.asc"

# === PARAMETERS ===
cellsize = 50  # meters per pixel in both X and Y

# === LOAD HEX GRID ===
gdf = gpd.read_file(hex_grid_path)

# Ensure CRS is projected
if gdf.crs is None:
    raise ValueError("Hex grid has no CRS defined.")

# === REPROJECT TO NETLOGO-COMPATIBLE CRS (EPSG:28992 assumed — adjust if needed) ===
target_crs = "EPSG:28992"
if gdf.crs.to_string() != target_crs:
    gdf = gdf.to_crs(target_crs)

# Get bounds
minx, miny, maxx, maxy = gdf.total_bounds

# Calculate raster dimensions based on new cell size
width = int((maxx - minx) / cellsize) + 1
height = int((maxy - miny) / cellsize) + 1

# Create transform
transform = from_origin(minx, maxy, cellsize, cellsize)

# Create shapes (geometry, value) pairs
shapes = ((geom, value) for geom, value in zip(gdf.geometry, gdf["tick_density"]))

# Save tick density as a Shapefile
shp_output_path = r"C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\shapefile\tick_density_ede_hexgrid.shp"
gdf.to_file(shp_output_path)

print(f"✅ Shapefile saved to: {shp_output_path}")


# Rasterize
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0.0,
    dtype="float32"
)

# Save as GeoTIFF
with rasterio.open(
    tif_output_path,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype='float32',
    crs=target_crs,
    transform=transform
) as dst:
    dst.write(raster, 1)

print(f"✅ Raster saved to: {tif_output_path}")

# === EXPORT TO ASCII GRID (.asc) with matching reference ===
from rasterio.transform import Affine

# Fixed values based on target .asc sample
ncols = 555
nrows = 360
xllcorner = 165928.79602051
yllcorner = 444768.2800293
cellsize = 50
nodata_val = -9999

# Define matching affine transform from lower-left corner
transform = Affine.translation(xllcorner, yllcorner + nrows * cellsize) * Affine.scale(cellsize, -cellsize)

# Create a new empty raster and rasterize into this fixed grid
raster = rasterize(
    shapes=shapes,
    out_shape=(nrows, ncols),
    transform=transform,
    fill=nodata_val,
    dtype="float32"
)

# Write to ASCII Grid
asc_profile = {
    'driver': 'AAIGrid',
    'height': nrows,
    'width': ncols,
    'count': 1,
    'dtype': 'float32',
    'transform': transform,
    'nodata': nodata_val,
    'crs': None  # AAIGrid doesn't support CRS
}

with rasterio.open(asc_output_path, 'w', **asc_profile) as asc_dst:
    asc_dst.write(raster, 1)

print(f"✅ ASCII Grid saved to: {asc_output_path}")


C:\Users\arman\AppData\Local\Temp\ipykernel_22696\3020321075.py:41: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(shp_output_path)
c:\Users\arman\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'tick_density' to 'tick_densi'
  ogr_write(


✅ Shapefile saved to: C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\shapefile\tick_density_ede_hexgrid.shp
✅ Raster saved to: C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede_50m.tif
✅ ASCII Grid saved to: C:\Users\arman\OneDrive - University of Twente\assignment_spatio\data\TickBites_Dec16\tick_density_ede_50m.asc
